In [28]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os
from typing import List, Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [30]:
def search_web_tool(query: str) -> str:
    tavily_key = os.getenv("TAVILY_API_KEY") 
    tool = TavilySearchResults(tavily_api_key=tavily_key)
    response = tool.invoke({"query":query})
    return response

def percentage_change_tool(start: float, end: float) -> float:
    return ((end - start) / start) * 100

In [31]:
planning_agent = AssistantAgent(
    "PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)

In [32]:
text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=10)
termination = text_mention_termination | max_messages_termination

In [33]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [34]:
team = SelectorGroupChat(
    [planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  # Allow an agent to speak multiple turns in a row.
)

In [35]:
task = "What was Italy's population in 2000 and 2024, and what is the percentage change between these two years?"

In [36]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
What was Italy's population in 2000 and 2024, and what is the percentage change between these two years?
---------- TextMessage (PlanningAgent) ----------
1. WebSearchAgent: Search for Italy's population in the year 2000.
2. WebSearchAgent: Search for Italy's population in the year 2024.
3. DataAnalystAgent: Calculate the percentage change in population from the year 2000 to the year 2024 using the data obtained.
---------- ToolCallRequestEvent (WebSearchAgent) ----------
[FunctionCall(id='call_1DRHhX7ymgYEqLuPU3Jbg8Xe', arguments='{"query": "Italy population in 2000"}', name='search_web_tool'), FunctionCall(id='call_aLqLNj9egOUiUjOkQybSkpwz', arguments='{"query": "Italy population in 2024"}', name='search_web_tool')]
---------- ToolCallExecutionEvent (WebSearchAgent) ----------
[FunctionExecutionResult(content="[{'title': 'Demographics of Italy - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Demographics_of_Italy', 'content': '| 1999 | 56,9

TaskResult(messages=[TextMessage(id='40faa5eb-2049-4645-9e5b-4fcfc157234c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 15, 21, 6, 19, 581088, tzinfo=datetime.timezone.utc), content="What was Italy's population in 2000 and 2024, and what is the percentage change between these two years?", type='TextMessage'), TextMessage(id='b2b07e81-1347-4442-9818-4359ebde9b7f', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=139, completion_tokens=66), metadata={}, created_at=datetime.datetime(2025, 7, 15, 21, 6, 22, 953720, tzinfo=datetime.timezone.utc), content="1. WebSearchAgent: Search for Italy's population in the year 2000.\n2. WebSearchAgent: Search for Italy's population in the year 2024.\n3. DataAnalystAgent: Calculate the percentage change in population from the year 2000 to the year 2024 using the data obtained.", type='TextMessage'), ToolCallRequestEvent(id='6e7e7046-28d8-4df2-8e90-42dcbf721c5e', source='WebSearchAgent', models_usag